# Tokenization

Write **"Wait, what?"** on paper and mark where you would cut it into pieces. A space gives you two pieces, `"Wait,"` and `"what?"`. If punctuation gets its own piece, you get four. A tokenizer makes that choice and gives each piece an integer ID.

The cut matters. It controls how many positions the model sees and which spellings share a piece. The ID itself is a lookup key. Token 12 is no closer in meaning to token 13 than it is to token 80.

## 1. Start with words

**Insight.** Splitting on whitespace is easy, but punctuation sticks to words and runs of whitespace disappear. If `"wait"` and `"wait!"` get different IDs, the model has to encounter both spellings to learn anything about both IDs.

**Implementation.** Split a line on whitespace. Then build a vocabulary from one short training sentence. Any piece missing from that vocabulary gets the same unknown ID.

In [1]:
training = "wait, what? wait"
words = training.split()
vocabulary = {"<UNK>": 0}
for word in words:
    if word not in vocabulary:
        vocabulary[word] = len(vocabulary)

for text in (training, "wait! what?", "wait    what?"):
    pieces = text.split()
    ids = [vocabulary.get(piece, vocabulary["<UNK>"]) for piece in pieces]
    print(repr(text), "→", pieces, "→", ids)

print("Vocabulary:", vocabulary)

'wait, what? wait' → ['wait,', 'what?', 'wait'] → [1, 2, 3]
'wait! what?' → ['wait!', 'what?'] → [0, 2]
'wait    what?' → ['wait', 'what?'] → [3, 2]
Vocabulary: {'<UNK>': 0, 'wait,': 1, 'what?': 2, 'wait': 3}


**Details.** Look at the third line. The four spaces leave no trace in the IDs. Joining the words with single spaces changes the input. The unknown ID also loses information because several different words can map to it.

## 2. Give punctuation and whitespace their own pieces

**Insight.** To reconstruct the original text, the tokenizer has to retain everything, including where the spaces and punctuation were.

**Implementation.** This toy rule groups runs of letters, digits, and underscores, groups whitespace, and keeps other symbols individually. Check the reconstructed string against the input.

In [2]:
import re

text = "Wait,  what?\nWait!"
pattern = r"\w+|\s+|[^\w\s]"
pieces = re.findall(pattern, text)
print([repr(piece) for piece in pieces])
assert "".join(pieces) == text

["'Wait'", "','", "'  '", "'what'", "'?'", "'\\n'", "'Wait'", "'!'"]


**Details.** This rule still treats every word as a separate vocabulary entry. It also splits `"can't"` into three pieces and keeps `"model42"` together. Those are consequences of the rule. Try both strings and inspect the pieces.

## 3. Use individual characters

**Insight.** A vocabulary can spell `"cats"` even if it only saw `"cat"` and `"dogs"` while training. It just needs the four individual characters. The cost is a longer token sequence.

**Implementation.** Build a character vocabulary from the training text and check which pieces it can encode.

In [3]:
training = "cat dogs"
characters = sorted(set(training))
char_to_id = {character: index for index, character in enumerate(characters)}
unknown_id = len(characters)

for text in ("cats", "cat!", "café", "🐈"):
    pieces = list(text)
    ids = [char_to_id.get(piece, unknown_id) for piece in pieces]
    print(repr(text), "→", pieces, "→", ids)

print("Unknown ID:", unknown_id)

'cats' → ['c', 'a', 't', 's'] → [2, 1, 7, 6]
'cat!' → ['c', 'a', 't', '!'] → [2, 1, 7, 8]
'café' → ['c', 'a', 'f', 'é'] → [2, 1, 8, 8]
'🐈' → ['🐈'] → [8]
Unknown ID: 8


**Details.** Python iterates through a string one Unicode code point at a time. What looks like one character on screen can contain several code points. The next cell compares two spellings of the same visible letter.

In [4]:
import unicodedata

composed = "é"
decomposed = "e\u0301"

for text in (composed, decomposed):
    print(repr(text), len(text), [f"U+{ord(character):04X}" for character in text])

assert composed != decomposed
assert unicodedata.normalize("NFC", decomposed) == composed

'é' 1 ['U+00E9']
'é' 2 ['U+0065', 'U+0301']


Normalization is a deliberate choice. It can make equivalent spellings share tokens. If you normalize before encoding, decoding recovers the normalized text.

## 4. Learn subword pieces with BPE

**Insight.** Common sequences can earn their own token. Rare words can still be assembled from smaller pieces. Byte-pair encoding (BPE) learns those larger pieces by repeatedly merging frequent adjacent pairs.

**Implementation.** Start with characters for this example so the merges are easy to read. Use a list of words so each word's boundary is explicit. Count pairs within each word.

In [5]:
from collections import Counter

def count_pairs(sequences):
    return Counter(
        pair
        for sequence in sequences
        for pair in zip(sequence, sequence[1:])
    )

words = ["play", "play", "played", "player", "replay"]
sequences = [list(word) for word in words]
print(count_pairs(sequences))

Counter({('p', 'l'): 5, ('l', 'a'): 5, ('a', 'y'): 5, ('y', 'e'): 2, ('e', 'd'): 1, ('e', 'r'): 1, ('r', 'e'): 1, ('e', 'p'): 1})


Apply one merge at a time and count again. The merge helper replaces nonoverlapping matches from left to right. For example, merging `a a` in `a a a` can replace only one pair.

In [6]:
from tokenizer import merge_pair

merges = []
for step in range(3):
    pair, frequency = count_pairs(sequences).most_common(1)[0]
    piece = "".join(pair)
    sequences = [merge_pair(sequence, pair, piece) for sequence in sequences]
    merges.append((pair, piece))
    print(f"{step + 1}. {pair} occurred {frequency} times → {piece!r}")
    print(sequences)

print("Overlap example:", merge_pair(list("aaa"), ("a", "a"), "aa"))

1. ('p', 'l') occurred 5 times → 'pl'
[['pl', 'a', 'y'], ['pl', 'a', 'y'], ['pl', 'a', 'y', 'e', 'd'], ['pl', 'a', 'y', 'e', 'r'], ['r', 'e', 'pl', 'a', 'y']]
2. ('pl', 'a') occurred 5 times → 'pla'
[['pla', 'y'], ['pla', 'y'], ['pla', 'y', 'e', 'd'], ['pla', 'y', 'e', 'r'], ['r', 'e', 'pla', 'y']]
3. ('pla', 'y') occurred 5 times → 'play'
[['play'], ['play'], ['play', 'e', 'd'], ['play', 'e', 'r'], ['r', 'e', 'play']]
Overlap example: ['aa', 'a']


**Details.** This implementation breaks ties by picking the pair encountered first. At encoding time, apply the learned merges in training order. The word `"playing"` was absent from training, but its opening can use the same piece as `"played"`.

In [7]:
for word in ("playing", "replayed"):
    pieces = list(word)
    for pair, piece in merges:
        pieces = merge_pair(pieces, pair, piece)
    print(word, "→", pieces)
    assert "".join(pieces) == word

playing → ['play', 'i', 'n', 'g']
replayed → ['r', 'e', 'play', 'e', 'd']


## 5. Look at the choices around the tokenizer

**Insight.** The same BPE algorithm can produce different vocabularies if you change the input before counting pairs. In particular, splitting first decides which boundaries a merge may cross.

**Implementation.** Compare counting on a whole string with counting separately inside the pieces produced by our earlier rule.

In [8]:
text = "go home go home"
whole_counts = count_pairs([list(text)])
split_counts = count_pairs([list(piece) for piece in re.findall(pattern, text)])

for pair in (("g", "o"), ("o", " "), (" ", "h")):
    print(pair, "whole string:", whole_counts[pair], "split first:", split_counts[pair])

('g', 'o') whole string: 2 split first: 2
('o', ' ') whole string: 2 split first: 0
(' ', 'h') whole string: 2 split first: 0


**Details.** A pre-tokenizer makes the initial cuts. BPE then learns merges within those cuts. Some rules attach a leading space to a word, so `"home"` and `" home"` can end up with different token IDs. Models may also reserve special IDs for markers such as end of sequence or padding. A reserved ID has a meaning defined by the tokenizer and model.

## 6. Finish at the byte level

**Insight.** The character vocabulary ran out of IDs for `"é"` and `"🐈"`. UTF-8 can encode both with the same 256 possible byte values it uses for everything else. Start BPE from those bytes and every valid UTF-8 string has a representation before any merges are learned.

**Implementation.** Convert a few strings to bytes. Compare the number of Python string elements with the number of bytes.

In [9]:
for text in ("A", "é", "你好", "🐈"):
    raw = text.encode("utf-8")
    print(repr(text), "code points:", len(text), "byte IDs:", list(raw))
    assert raw.decode("utf-8") == text

'A' code points: 1 byte IDs: [65]
'é' code points: 1 byte IDs: [195, 169]
'你好' code points: 2 byte IDs: [228, 189, 160, 229, 165, 189]
'🐈' code points: 1 byte IDs: [240, 159, 144, 136]


Now use the repository's byte-level BPE implementation. Open `tokenizer.py` alongside this cell. `train` counts pairs and assigns new IDs starting at 256. `encode` applies the stored merges. `decode` expands each ID into bytes, joins the bytes, then decodes the whole result as UTF-8.

In [10]:
from tokenizer import BytePairTokenizer

tokenizer = BytePairTokenizer()
training = "banana banana banana"
example = "banana 🍌"

print("Before training:", tokenizer.encode(example))
tokenizer.train(training, merge_count=2)

for pair, token_id in tokenizer.merges:
    print(pair, "→", token_id, "contains", tokenizer.vocabulary[token_id])

ids = tokenizer.encode(example)
print("After training:", ids)
print("Decoded:", repr(tokenizer.decode(ids)))
assert tokenizer.decode(ids) == example

Before training: [98, 97, 110, 97, 110, 97, 32, 240, 159, 141, 140]
(97, 110) → 256 contains b'an'
(98, 256) → 257 contains b'ban'
After training: [257, 256, 97, 32, 240, 159, 141, 140]
Decoded: 'banana 🍌'


**Details.** The banana emoji still takes four IDs because the training string contained none of its bytes. Those IDs may each represent an incomplete UTF-8 fragment. Look up their bytes one by one, but decode them together.

In [11]:
emoji_ids = tokenizer.encode("🍌")
fragments = [tokenizer.vocabulary[token_id] for token_id in emoji_ids]
print("Fragments:", fragments)
print("Joined and decoded:", b"".join(fragments).decode("utf-8"))

for text in ("", " café 你好 🍌 ", "one\n\ttwo", decomposed):
    ids = tokenizer.encode(text)
    assert tokenizer.decode(ids) == text
    print(repr(text), len(ids), "IDs")

Fragments: [b'\xf0', b'\x9f', b'\x8d', b'\x8c']
Joined and decoded: 🍌
'' 0 IDs
' café 你好 🍌 ' 19 IDs
'one\n\ttwo' 8 IDs
'é' 3 IDs


## Try a different training corpus

Predict which text will benefit from each vocabulary, then run this cell. Change `merge_budget` or add a test string. The counts show how the training text changes the allocation of tokens.

In [12]:
corpora = {
    "fruit": "banana banana banana banana",
    "code": "print(value)\n" * 4,
}
examples = ("banana 🍌", "print(value)", "你好")
merge_budget = 8

print(f"{'training text':<14}" + "".join(f"{text!r:>17}" for text in examples))
for label, corpus in corpora.items():
    tokenizer = BytePairTokenizer()
    tokenizer.train(corpus, merge_budget)
    encoded = [tokenizer.encode(text) for text in examples]
    assert [tokenizer.decode(ids) for ids in encoded] == list(examples)
    print(f"{label:<14}" + "".join(f"{len(ids):>17}" for ids in encoded))

training text        'banana 🍌'   'print(value)'             '你好'
fruit                         5               12                6
code                         11                4                6
